# Phase 4 — Unified Evaluation & Comparison

**Description:** Load all fine-tuned models (ViT5-base, BARTpho-word) and the zero-shot Groq baseline. Run batch inference on the unseen test set, compute full metrics (ROUGE, BLEU, BERTScore), and export the comparison table.

**Hardware:** T4 GPU or higher is highly recommended for faster inference.
**Outputs:**
- `results/comparison_table.csv`
- `results/per_topic_analysis.csv`
- `results/eval_vit5.json`
- `results/eval_bartpho.json`

## Cell 1 — Setup & Install Dependencies

In [3]:
import os, sys

# ── Detect platform & Mount Drive ──
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/vimedaq-project"
elif ON_KAGGLE:
    # Assuming you upload the dataset/models appropriately to Kaggle
    DRIVE_ROOT = "/kaggle/working"
else:
    DRIVE_ROOT = ".."

print(f"Drive root set to: {DRIVE_ROOT}")

# Install dependencies (quietly)
!pip install transformers==4.46.3 evaluate rouge-score sacrebleu bert-score sentencepiece pyvi -q

Mounted at /content/drive
Drive root set to: /content/drive/MyDrive/vimedaq-project


## Cell 2 — Load Test Data

In [4]:
import torch
import json
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
from pyvi import ViTokenizer
from tqdm import tqdm

test_path = os.path.join(DRIVE_ROOT, "data", "processed", "test.json")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Test file not found at {test_path}")

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

questions  = [d['question'] for d in test_data]
contexts   = [d['context']  for d in test_data]
references = [d['answer']   for d in test_data]
topics     = [d['topic']    for d in test_data]

print(f"Test samples loaded: {len(test_data)}")

Test samples loaded: 2348


## Cell 3 — Batch Inference Function

In [8]:
def batch_inference(model, tokenizer, questions, contexts,
                    max_input=512, max_target=128, batch_size=8, device='cuda', use_pyvi=False):
    """Run batch inference, return list of predicted strings."""
    model.eval()
    model.to(device)
    predictions = []

    # Tích hợp PyVi cho BARTpho nếu use_pyvi=True
    if use_pyvi:
        from pyvi import ViTokenizer

    for i in range(0, len(questions), batch_size):
        batch_q = questions[i:i+batch_size]
        batch_c = contexts[i:i+batch_size]

        if use_pyvi:
            batch_q = [ViTokenizer.tokenize(q) for q in batch_q]
            batch_c = [ViTokenizer.tokenize(c) for c in batch_c]

        inputs_text = [f"question: {q} context: {c}" for q, c in zip(batch_q, batch_c)]

        inputs = tokenizer(inputs_text, return_tensors="pt",
                           max_length=max_input, padding=True, truncation=True).to(device)

        # --- ĐOẠN CODE KHẮC PHỤC LỖI ---
        # Loại bỏ token_type_ids nếu tokenizer tự động sinh ra
        if "token_type_ids" in inputs:
            del inputs["token_type_ids"]
        # -------------------------------

        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_target,
                                      num_beams=4, early_stopping=True)

        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend([p.strip() for p in preds])

    return predictions

## Cell 4 — Evaluate ViT5-base

In [6]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

vit5_path = os.path.join(DRIVE_ROOT, "checkpoints", "vit5", "best")
print(f"Loading ViT5 from {vit5_path}...")
vit5_tokenizer = AutoTokenizer.from_pretrained(vit5_path)
vit5_model     = AutoModelForSeq2SeqLM.from_pretrained(vit5_path)

vit5_preds = batch_inference(vit5_model, vit5_tokenizer, questions, contexts,
                             max_input=512, max_target=128, batch_size=8, device=DEVICE, use_pyvi=False)
print("ViT5 inference done.")

# Free memory
del vit5_model
torch.cuda.empty_cache()

Using device: cuda
Loading ViT5 from /content/drive/MyDrive/vimedaq-project/checkpoints/vit5/best...


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
Inference: 100%|██████████| 294/294 [10:01<00:00,  2.05s/it]

ViT5 inference done.


## Cell 5 — Evaluate BARTpho-word

In [9]:
bartpho_path = os.path.join(DRIVE_ROOT, "checkpoints", "bartpho", "best")
print(f"Loading BARTpho from {bartpho_path}...")
bartpho_tokenizer = AutoTokenizer.from_pretrained(bartpho_path)
bartpho_model     = AutoModelForSeq2SeqLM.from_pretrained(bartpho_path)

bartpho_preds = batch_inference(bartpho_model, bartpho_tokenizer, questions, contexts,
                                max_input=1024, max_target=256, batch_size=8, device=DEVICE, use_pyvi=True)
print("BARTpho inference done.")

# Free memory
del bartpho_model
torch.cuda.empty_cache()

Loading BARTpho from /content/drive/MyDrive/vimedaq-project/checkpoints/bartpho/best...
BARTpho inference done.


## Cell 6 — Compute All Metrics

In [10]:
rouge  = evaluate.load("rouge")
bleu   = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")

def compute_all_metrics(preds, refs, model_name):
    rouge_r   = rouge.compute(predictions=preds, references=refs)
    bleu_r    = bleu.compute(predictions=preds, references=[[r] for r in refs])
    bs_r      = bertscore.compute(predictions=preds, references=refs, lang="vi")
    return {
        "model":    model_name,
        "rouge1":   round(rouge_r['rouge1'], 4),
        "rouge2":   round(rouge_r['rouge2'], 4),
        "rougeL":   round(rouge_r['rougeL'], 4),
        "bleu4":    round(bleu_r['bleu'], 4),
        "bertscore_f1": round(float(np.mean(bs_r['f1'])), 4),
    }

results = []
print("Computing metrics for ViT5...")
results.append(compute_all_metrics(vit5_preds, references, "ViT5-base (fine-tuned)"))

print("Computing metrics for BARTpho...")
results.append(compute_all_metrics(bartpho_preds, references, "BARTpho-word (fine-tuned)"))

# Add Groq baseline from saved file
groq_baseline_path = os.path.join(DRIVE_ROOT, "results", "baseline_groq.json")
if os.path.exists(groq_baseline_path):
    with open(groq_baseline_path, "r", encoding="utf-8") as f:
        groq_res = json.load(f)
    results.append({
        "model": "Llama-3.3-70B-versatile (zero-shot)",
        "rouge1": groq_res.get('rouge1', "N/A"),
        "rouge2": groq_res.get('rouge2', "N/A"),
        "rougeL": groq_res.get('rougeL', "N/A"),
        "bleu4": groq_res.get('bleu4', "N/A"),
        "bertscore_f1": groq_res.get('bertscore_f1', "N/A")
    })
else:
    print(f"Warning: {groq_baseline_path} not found. Skipping Groq baseline.")

comparison_df = pd.DataFrame(results)
print("\n" + "="*60)
print(comparison_df.to_string(index=False))
print("="*60)

Computing metrics for ViT5...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Computing metrics for BARTpho...

                              model  rouge1  rouge2  rougeL  bleu4  bertscore_f1
             ViT5-base (fine-tuned)  0.7318  0.6153  0.6680 0.5005        0.8752
          BARTpho-word (fine-tuned)  0.7327  0.6185  0.6681 0.2794        0.8230
Llama-3.3-70B-versatile (zero-shot)  0.7285  0.6257  0.6664 0.4684        0.8757


## Cell 7 — Per-topic Analysis

In [11]:
# Per-topic ROUGE-L for ViT5 (most granular analysis)
topic_results = []
unique_topics = list(set(topics))

for topic in unique_topics:
    idx = [i for i, t in enumerate(topics) if t == topic]
    t_preds = [vit5_preds[i] for i in idx]
    t_refs  = [references[i]  for i in idx]
    r = rouge.compute(predictions=t_preds, references=t_refs)
    topic_results.append({"topic": topic, "n": len(idx), "rougeL": round(r['rougeL'], 4)})

topic_df = pd.DataFrame(topic_results)
print("\nPer-topic ROUGE-L (ViT5):")
print(topic_df.to_string(index=False))


Per-topic ROUGE-L (ViT5):
 topic   n  rougeL
     0 229  0.6528
     1 780  0.6274
     2 588  0.7007
     3 751  0.6920


## Cell 8 — Save All Results

In [12]:
results_dir = os.path.join(DRIVE_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)

# Comparison tables
comparison_df.to_csv(os.path.join(results_dir, "comparison_table.csv"), index=False)
topic_df.to_csv(os.path.join(results_dir, "per_topic_analysis.csv"), index=False)

# Detailed predictions for each model
for model_name, preds in [("vit5", vit5_preds), ("bartpho", bartpho_preds)]:
    out = [{"question": questions[i], "reference": references[i],
            "prediction": preds[i], "topic": topics[i]}
           for i in range(len(preds))]
    with open(os.path.join(results_dir, f"eval_{model_name}.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

print("✅ All evaluation results saved to Drive.")
print("Phase 4 is complete!")

✅ All evaluation results saved to Drive.
Phase 4 is complete!
